In [ ]:
pip install torch --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 680.3 kB/s eta 0:00:0000:0100:01
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 19.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 3.8 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 24.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 665.5 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 4.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 10.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━

In [ ]:
%pip install neuralforecast

  Using cached optuna-4.8.0-py3-none-any.whl.metadata (17 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached msgpack-1.1.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cached protobuf-7.34.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
  Using cached tensorboardx-2.6.5-py3-none-any.whl.metadata (6.6 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached sqlalchemy-2.0.49-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (9.5 kB)
  Using cached aiohttp-3.13.5-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 

In [ ]:
import time
import numpy as np
import pandas as pd
import json

from sklearn.metrics import mean_absolute_error, mean_squared_error

# === CarbonTracker ===
import io
import re
from contextlib import redirect_stdout
from carbontracker.tracker import CarbonTracker as Tracker

# === Eco2AI ===
import eco2ai
eco2ai_tracker = eco2ai.Tracker(project_name="eco2ai", file_name="eco2ai_emissions_dl.csv", ignore_warnings=True)

/home/enginelab/Neurips 2026/.venv/lib/python3.12/site-packages/carbontracker/components/gpu/nvidia.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml


In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import TCN, NBEATS, Autoformer

/home/enginelab/Neurips 2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-04 06:50:25,215	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-05-04 06:50:25,353	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [ ]:
class CarbonTracker:
  def __init__(self, epochs=1, verbose=1):
    self.epochs = epochs
    self.verbose = verbose
    self.buffer = io.StringIO()
    self.tracker = None
    self.co2_g = None
    self.energy_kwh = None

  def __enter__(self):
    self._stdout_ctx = redirect_stdout(self.buffer)
    self._stdout_ctx.__enter__()
    self.tracker = Tracker(epochs=self.epochs, verbose=self.verbose)
    self.tracker.epoch_start()
    return self

  def __exit__(self, exc_type, exc, tb):
    self.tracker.epoch_end()
    self.tracker.stop()
    self._stdout_ctx.__exit__(exc_type, exc, tb)
    output = self.buffer.getvalue()

    match_co2 = re.search(r"CO2eq:\s*([\d\.]+)\s*g", output)
    self.co2_g = float(match_co2.group(1)) if match_co2 else None

    match_energy = re.search(r"Energy:\s*([\d\.]+)\s*kWh", output)
    self.energy_kwh = float(match_energy.group(1)) if match_energy else None

In [ ]:
def load_series(path, unique_id="series_1"):
    df = pd.read_csv(path, index_col='date')
    df = df.reset_index().rename(columns={'date': 'ds', 'value': 'y'})
    df['unique_id'] = unique_id
    df["ds"] = pd.to_datetime(df["ds"])    # Precisei adicionar pra ML

    return df

def smape(y_true, y_pred, decimals: int = 2) -> float:
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    return round(float(np.mean(numerator / (denominator + 1e-10)) * 100), decimals)

def run_forecast(train_df, true_df, freq, season_length, dataset_name):
  results = []

  input_size = len(train_df) - 1
  horizon = len(true_df)

  models = [
        ("TCN", TCN(input_size=input_size, h=horizon, max_steps=700)),
        ("NBEATS", NBEATS(input_size=input_size, h=horizon, max_steps=700)),
        ("AUTOFORMER", Autoformer(input_size=input_size, h=horizon, max_steps=700))
  ]

  for model_name, model in models:

    print("Rodando Modelo: ", model_name, "pro Dataset: ", dataset_name)

    nf = NeuralForecast(
      models=[model],
      freq=freq
    )

    eco2ai_tracker.start()

    with CarbonTracker() as carbontracker:
      start = time.time()

      nf.fit(train_df)
      forecast = nf.predict(h=len(true_df))

      end = time.time()

    eco2ai_tracker.stop()

    y_true = true_df['y'].values
    y_pred = forecast[model_name].values

    carbontracker_g = carbontracker.co2_g
    carbontracker_kWh = carbontracker.energy_kwh
    carbontracker_Wh = carbontracker_kWh * 1000 if carbontracker_kWh else None

    eco2ai_data = pd.read_csv("eco2ai_emissions_dl.csv").iloc[-1]
    eco2ai_kWh = eco2ai_data["power_consumption(kWh)"]
    eco2ai_Wh = eco2ai_kWh * 1000
    eco2ai_kg = eco2ai_data["CO2_emissions(kg)"]
    eco2ai_g = eco2ai_kg * 1000

    results.append({
      "Dataset": dataset_name,
      "Modelo": model_name,
      "Parâmetros": str(model),
      "SMAPE": smape(y_true, y_pred),
      "MAE": mean_absolute_error(y_true, y_pred),
      "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
      "CarbonTracker CO₂ (g)": carbontracker_g,
      "CarbonTracker Consumo (Wh)": carbontracker_Wh,
      "Eco2AI CO₂ (g)": eco2ai_g,
      "Eco2AI Consumo (Wh)": eco2ai_Wh,
      "Tempo (s)": end - start,
      "y_true": json.dumps(y_true.tolist()),
      "y_pred": json.dumps(y_pred.tolist())
    })

  return results

/home/enginelab/Neurips 2026/.venv/lib/python3.12/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/enginelab/Neurips 2026/.venv/lib/python3.12/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '620.2497460842133' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/enginelab/Neurips 2026/.venv/lib/python3.12/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.0025472324540

In [ ]:
data_path = "../datasets modificados"

In [ ]:
etth2 = load_series(f'{data_path}/wavelet_whiteNoise_ETTh2.csv', 'etth2')
electricity = load_series(f'{data_path}/wavelet_whiteNoise_electricity.csv', 'electricity')
traffic = load_series(f'{data_path}/magnitudeWarp_traffic.csv', 'traffic')
covid19 = load_series(f'{data_path}/shifted_Covid-19.csv', 'covid19')
retail = load_series(f'{data_path}/scaling_retail.csv', 'retail')
wike2000 = load_series(f'{data_path}/scaling_wike2000.csv', 'wike2000')
carbon = load_series(f'{data_path}/carbon.csv', 'carbon')

In [ ]:
etth2_train = etth2[(etth2['ds'] >= '2016-07-12 06:00:00') & (etth2['ds'] <= '2016-09-20 05:00:00')]
etth2_true = etth2[etth2['ds'] > '2016-09-20 05:00:00'].iloc[:24]

electricity_train = electricity[(electricity['ds'] >= '2013-06-28 07:00:01') & (electricity['ds'] <= '2013-09-06 06:00:01')]
electricity_true = electricity[electricity['ds'] > '2013-09-06 06:00:01'].iloc[:24]

traffic_train = traffic[(traffic['ds'] >= '2018-01-10 03:00:00') & (traffic['ds'] <= '2018-03-21 02:00:00')]
traffic_true = traffic[traffic['ds'] > '2018-03-21 02:00:00'].iloc[:24]

covid19_train = covid19[(covid19['ds'] >= '2021-03-12') & (covid19['ds'] <= '2022-03-10')]
covid19_true = covid19[covid19['ds'] > '2022-03-10'].iloc[:7]


retail_train = retail[(retail['ds'] >= '2017-01-01') & (retail['ds'] <= '2017-12-30')]
retail_true = retail[retail['ds'] > '2017-12-30'].iloc[:7]

wike2000_train = wike2000[(wike2000['ds'] >= '2012-07-08') & (wike2000['ds'] <= '2013-07-06')]
wike2000_true = wike2000[wike2000['ds'] > '2013-07-06'].iloc[:7]

carbon_train = carbon[(carbon['ds'] >= '2025-02-14') & (carbon['ds'] <= '2026-02-12')]
carbon_true = carbon[carbon['ds'] > '2026-02-12'].iloc[:7]

In [ ]:
etth2_train

,ds,y,unique_id
0,2016-07-12 06:00:00,37.224386,etth2
1,2016-07-12 07:00:00,36.876556,etth2
2,2016-07-12 08:00:00,37.694396,etth2
3,2016-07-12 09:00:00,39.848802,etth2
4,2016-07-12 10:00:00,41.690677,etth2
...,...,...,...
1675,2016-09-20 01:00:00,28.586860,etth2
1676,2016-09-20 02:00:00,28.018986,etth2
1677,2016-09-20 03:00:00,27.641455,etth2
1678,2016-09-20 04:00:00,26.437257,etth2


/home/enginelab/Neurips 2026/.venv/lib/python3.12/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/enginelab/Neurips 2026/.venv/lib/python3.12/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '380.2461738586426' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/enginelab/Neurips 2026/.venv/lib/python3.12/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.0016307111833

In [ ]:
all_results = []

all_results += run_forecast(etth2_train, etth2_true, freq='h', season_length=24, dataset_name='ETTh2')
all_results += run_forecast(electricity_train, electricity_true, freq='h', season_length=24, dataset_name='Electricity')
all_results += run_forecast(traffic_train, traffic_true, freq='h', season_length=24, dataset_name='Traffic')
all_results += run_forecast(covid19_train, covid19_true, freq='d', season_length=7, dataset_name='Covid-19')
all_results += run_forecast(retail_train, retail_true, freq='d', season_length=7, dataset_name='Retail')
all_results += run_forecast(wike2000_train, wike2000_true, freq='d', season_length=7, dataset_name='Wike2000')
all_results += run_forecast(carbon_train, carbon_true, freq='d', season_length=7, dataset_name='Carbon')

Seed set to 1
Seed set to 1
Seed set to 1


Rodando Modelo:  TCN pro Dataset:  ETTh2


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type                       | Params | Mode  | FLOPs
-------------------------------------------------------------------------------
0 | loss            | MAE                        | 0      | train | 0    
1 | padder_train    | ConstantPad1d              | 0      | train | 0    
2 | scaler          | TemporalNorm               | 0      | train | 0    
3 | hist_encoder    | TemporalConvolutionEncoder | 131 K  | train | 0    
4 | context_adapter | Linear                     | 40.3 K | train | 0    
5 | mlp_decoder     | MLP                        | 16.6 K | train | 0    
-------------------------------------------------

Rodando Modelo:  NBEATS pro Dataset:  ETTh2


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type          | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | loss         | MAE           | 0      | train | 0    
1 | padder_train | ConstantPad1d | 0      | train | 0    
2 | scaler       | TemporalNorm  | 0      | train | 0    
3 | blocks       | ModuleList    | 6.0 M  | train | 0    
---------------------------------------------------------------
5.9 M     Trainable params
83.4 K    Non-trainable params
6.0 M     Total params
23.807    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode
0         Total Flops
Error submitti

Rodando Modelo:  AUTOFORMER pro Dataset:  ETTh2


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type          | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | loss          | MAE           | 0      | train | 0    
1 | padder_train  | ConstantPad1d | 0      | train | 0    
2 | scaler        | TemporalNorm  | 0      | train | 0    
3 | decomp        | SeriesDecomp  | 0      | train | 0    
4 | enc_embedding | DataEmbedding | 384    | train | 0    
5 | dec_embedding | DataEmbedding | 384    | train | 0    
6 | encoder       | Encoder       | 148 K  | train | 0    
7 | decoder       | Decoder       | 141 K  | train | 0    
---------------------------------------------------

OutOfMemoryError: CUDA out of memory. Tried to allocate 840.00 MiB. GPU 0 has a total capacity of 19.58 GiB of which 416.19 MiB is free. Process 928019 has 1.41 GiB memory in use. Including non-PyTorch memory, this process has 17.73 GiB memory in use. Of the allocated memory 17.48 GiB is allocated by PyTorch, and 8.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

Error submitting job "Tracker._func_for_sched (trigger: interval[0:00:10], next run at: 2026-05-04 07:04:28 -03)" to executor "default"
Traceback (most recent call last):
  File "/home/enginelab/Neurips 2026/.venv/lib/python3.12/site-packages/apscheduler/schedulers/base.py", line 1195, in _process_jobs
    executor.submit_job(job, run_times)
  File "/home/enginelab/Neurips 2026/.venv/lib/python3.12/site-packages/apscheduler/executors/base.py", line 74, in submit_job
    self._do_submit_job(job, run_times)
  File "/home/enginelab/Neurips 2026/.venv/lib/python3.12/site-packages/apscheduler/executors/pool.py", line 27, in _do_submit_job
    f = self._pool.submit(
        ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/thread.py", line 170, in submit
    raise RuntimeError('cannot schedule new futures after shutdown')
RuntimeError: cannot schedule new futures after shutdown
Error submitting job "Tracker._func_for_sched (trigger: interval[0:00:10], next run at: 2026-05-04 

In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_csv("../experiments/deep learning/experimentos_dl.csv", index=False)
results_df

,Dataset,Modelo,Parâmetros,SMAPE,MAE,RMSE,CarbonTracker CO₂ (g),CarbonTracker Consumo (Wh),Eco2AI CO₂ (g),Eco2AI Consumo (Wh),Tempo (s),y_true,y_pred
0,ETTh2,RandomForest,RandomForestRegressor(),20.96,6.771467,8.500905,0.001633,0.015823,0.002674,0.023653,0.448601,"[25.35650062561035, 25.136999130249023, 27.333...","[25.732299938201905, 25.72571994781494, 25.732..."
1,ETTh2,XGBoost,"XGBRegressor(base_score=None, booster=None, ca...",19.36,6.170565,8.037044,0.001573,0.015238,0.002532,0.022393,0.376004,"[25.35650062561035, 25.136999130249023, 27.333...","[25.038986206054688, 24.706754684448242, 24.89..."
2,ETTh2,CatBoost,"CatBoostRegressor(loss_function='RMSE', verbos...",15.96,5.232525,7.354678,0.002598,0.025170,0.003299,0.029180,0.584575,"[25.35650062561035, 25.136999130249023, 27.333...","[26.690334400514736, 27.273893988399145, 27.71..."
3,Electricity,RandomForest,RandomForestRegressor(),9.93,0.073954,0.149355,0.002471,0.023938,0.003204,0.028337,0.547240,"[-1.376244106473734, -1.2267854542201622, -0.8...","[-1.403984393904747, -1.209235962154592, -0.18..."
4,Electricity,XGBoost,"XGBRegressor(base_score=None, booster=None, ca...",21.05,0.180516,0.269628,0.000668,0.006472,0.001998,0.017670,0.152892,"[-1.376244106473734, -1.2267854542201622, -0.8...","[-1.401699185371399, -1.2174557447433472, 0.07..."
5,Electricity,CatBoost,"CatBoostRegressor(loss_function='RMSE', verbos...",10.35,0.105156,0.131261,0.002230,0.021602,0.003017,0.026684,0.496268,"[-1.376244106473734, -1.2267854542201622, -0.8...","[-1.3526986956722638, -1.188026044321824, -0.5..."
6,Transporte,RandomForest,RandomForestRegressor(),13.27,76.507500,132.846114,0.002079,0.020139,0.002930,0.025909,0.472237,"[680.0, 810.0, 1131.0, 1346.0, 1194.0, 792.0, ...","[693.35, 733.62, 1016.1, 1374.54, 1131.99, 831..."
7,Transporte,XGBoost,"XGBRegressor(base_score=None, booster=None, ca...",38.54,76.619410,138.746706,0.000504,0.004881,0.001818,0.016080,0.112118,"[680.0, 810.0, 1131.0, 1346.0, 1194.0, 792.0, ...","[642.0361938476562, 725.8562622070312, 1014.60..."
8,Transporte,CatBoost,"CatBoostRegressor(loss_function='RMSE', verbos...",38.45,79.228495,137.304024,0.002253,0.021826,0.003056,0.027024,0.521380,"[680.0, 810.0, 1131.0, 1346.0, 1194.0, 792.0, ...","[658.7804124714141, 731.8366368922091, 1031.42..."
9,Covid-19,RandomForest,RandomForestRegressor(),48.80,2176.412857,2662.503682,0.000515,0.004988,0.001791,0.015843,0.114476,"[4184, 4194, 3614, 3116, 2503, 2568, 2876]","[4338.7, 4479.43, 4883.33, 5458.16, 5674.89, 6..."
